# AstroTutor — Valutazione a tre bracci su Colab

Rigenera **tutte** le risposte e le rigiudica con un modello più grande, in modo che i tre
bracci (`qwen2.5:3b`, `astrotutor-dpo`, `astrotutor-dpo-v2`) siano misurati nelle identiche
condizioni.

Rispetto alla run locale cambiano tre cose:

| | Locale | Qui |
|---|---|---|
| Giudice | `qwen2.5:7b-instruct` al 55% su CPU | **`qwen2.5:14b-instruct` al 100% GPU** |
| Encoder (BGE-m3 + re-ranker) | CPU | **GPU** |
| Generazione | `temperature=0.1`, nessun seed | **`temperature=0`, seed fisso** → riproducibile |

Tempo stimato: **~1 ora** su L4 (contro le 4+ ore in locale). Circa 5 crediti.

Il notebook **non modifica il repository**: applica le patch alla copia scompattata in
`/content`, lasciando intatti i sorgenti originali.

## Prima di iniziare

Prepara due file su Drive in `MyDrive/astrotutor/`:

**1. `astrotutor.zip`** — il repo senza le cartelle pesanti. In locale, da PowerShell nella
radice del progetto:

```powershell
$dst = "$env:TEMP\astrotutor_pack"
Remove-Item $dst -Recurse -Force -ErrorAction SilentlyContinue
robocopy . $dst /E /XD .git .venv data\raw data\vector_db models __pycache__ /NFL /NDL /NJH /NJS
Compress-Archive -Path "$dst\*" -DestinationPath "$HOME\Desktop\astrotutor.zip" -Force
```

**2. `models/astrotutor-3b-dpo-Q4_K_M.gguf`** — il GGUF del modello a 378 triplette (1,8 GB),
che serve per il secondo braccio. Il `-v2` dovrebbe gia' essere li' dal training.

## 1 — Mount di Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 — Verifica dei prerequisiti

Controlla che i file su Drive ci siano tutti **prima** di installare 10 GB di roba.

In [ ]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/astrotutor")
ZIP   = DRIVE / "astrotutor.zip"
GGUF_OLD = DRIVE / "models" / "astrotutor-3b-dpo-Q4_K_M.gguf"
GGUF_V2  = DRIVE / "models" / "astrotutor-3b-dpo-v2-Q4_K_M.gguf"

ok = True
for f in (ZIP, GGUF_OLD, GGUF_V2):
    if f.exists():
        print(f"OK   {f.name:40s} {f.stat().st_size/1e6:8.0f} MB")
    else:
        print(f"MANCA {f}")
        ok = False

# il GGUF del v2 potrebbe avere ancora il nome originale prodotto dal training
if not GGUF_V2.exists():
    alt = DRIVE / "models" / "astrotutor-3b-dpo-Q4_K_M.gguf"
    print(f"\nSe il v2 su Drive si chiama ancora '{alt.name}', rinominalo in "
          f"'{GGUF_V2.name}' oppure aggiorna GGUF_V2 qui sopra.")

assert ok, "Carica i file mancanti su Drive prima di proseguire"

import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)} — "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 3 — Setup: dipendenze e Ollama

Installa le librerie Python e Ollama nel runtime, poi avvia il server in background.

In [ ]:
# Versioni pinnate a quelle dell'ambiente locale, dove la valutazione gira.
#
# Perche' servono TUTTE: ragas/llms/base.py importa langchain-openai e (dentro un
# try/except, ma solo dalle versioni recenti) langchain_community.chat_models.vertexai.
# Con un ragas piu' vecchio quell'import NON e' protetto e fallisce, perche' in
# langchain-community 0.4.x il modulo vertexai non esiste piu'. E' esattamente
# l'errore che si vede alla cella 8 se si lascia pip libero di risolvere.
#
# NON serve google-cloud-aiplatform: e' Vertex AI, qui il giudice gira su Ollama.
!pip install -q     "ragas==0.4.3"     "langchain-core==1.5.0"     "langchain-openai==1.4.0"     "langchain-community==0.4.2"     "openai==2.46.0"     "instructor==1.15.4"     chromadb sentence-transformers python-dotenv rank_bm25

import importlib.metadata as md
attese = {"ragas": "0.4.3", "langchain-core": "1.5.0", "langchain-openai": "1.4.0",
          "langchain-community": "0.4.2", "openai": "2.46.0", "instructor": "1.15.4"}
problemi = []
for pkg, atteso in attese.items():
    try:
        v = md.version(pkg)
    except Exception:
        v = None
    stato = "OK " if v == atteso else "!! "
    if v != atteso:
        problemi.append(f"{pkg}: atteso {atteso}, trovato {v}")
    print(f"{stato}{pkg:22s} {v}")

if problemi:
    print("
ATTENZIONE — versioni diverse dall'ambiente di riferimento:")
    for x in problemi:
        print("   ", x)
    print("Rilancia questa cella; se persiste, la cella 8 fallira' sull'import di ragas.")

⚠️ **Dopo questa cella: `Runtime > Riavvia sessione`.**

pip ha sostituito pacchetti che Colab aveva gia' caricato in memoria. Il riavvio della
sessione **non cancella i file** in `/content` (solo un "Disconnetti ed elimina runtime"
lo farebbe), quindi non perdi niente: riparti dalla cella 1 e le celle pesanti
(download dei modelli, indice) trovano tutto gia' fatto.

In [ ]:
import subprocess, time, os, shutil, requests

# L'installer di Ollama estrae l'archivio con zstd, che l'immagine Colab non ha:
# senza, l'installazione fallisce in silenzio e il binario non viene creato.
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

assert shutil.which("ollama"), "installazione di Ollama fallita: rileggi l'output qui sopra"
!ollama --version

# Due modelli residenti insieme (generatore + giudice): senza questo Ollama ne
# scarica uno a ogni alternanza, e in questa valutazione si alternano a ogni domanda.
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "2"
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"

subprocess.Popen(["ollama", "serve"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                 env={**os.environ})

for i in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2)
        print(f"Ollama pronto dopo {i}s")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama non risponde su :11434")

## 4 — Configurazione

Tutti i parametri della run stanno qui. Se cambi il giudice, cambiano **tutti** i numeri:
non sono confrontabili con quelli di una run fatta con un giudice diverso.

In [ ]:
import shutil, os, zipfile
from pathlib import Path

# --- Parametri della valutazione ---
JUDGE_MODEL   = "qwen2.5:14b-instruct"   # ~9 GB in Q4: sta su L4 insieme al resto.
                                         # Su A100 puoi salire a "qwen2.5:32b-instruct".
GEN_TEMPERATURE = "0.0"                  # 0 + seed = risposte riproducibili
GEN_SEED        = "42"
EMBED_DEVICE    = "cuda"                 # BGE-m3
RERANK_DEVICE   = "cuda"                 # cross-encoder: e' il piu' esigente
MODELS = ["qwen2.5:3b", "astrotutor-dpo", "astrotutor-dpo-v2"]

# --- Scompatta il repo (copia di lavoro: l'originale su Drive non viene toccato) ---
# IDEMPOTENTE: se il repo c'e' gia' non lo ritocca. L'indice ChromaDB viene scritto
# dentro il repo scompattato (data/vector_db), quindi riestrarre lo cancellerebbe e
# costringerebbe a rifare la cella 7. Metti RIESTRAI=True solo se hai cambiato lo zip.
RIESTRAI = False

DEST = Path("/content/unzip")
gia_presente = bool(list(DEST.rglob("src/generation.py"))) if DEST.exists() else False

if gia_presente and not RIESTRAI:
    print("Repo gia' presente: non riestraggo (RIESTRAI=True per forzare)")
else:
    # Compress-Archive di PowerShell 5.1 scrive i percorsi interni con "\\" invece di
    # "/": shutil.unpack_archive li interpreta come nomi di file e non ricrea le
    # cartelle. Scompattiamo a mano normalizzando i separatori.
    shutil.rmtree(DEST, ignore_errors=True)
    DEST.mkdir(parents=True)
    with zipfile.ZipFile(ZIP) as z:
        for info in z.infolist():
            nome = info.filename.replace("\\", "/")
            if nome.endswith("/"):
                continue
            target = DEST / nome
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as fsrc, open(target, "wb") as fdst:
                shutil.copyfileobj(fsrc, fdst)

# Lo zip puo' avere i file alla radice o dentro una cartella di primo livello:
# individuiamo la radice vera cercando un file che sappiamo esserci.
trovati = list(DEST.rglob("src/generation.py"))
assert trovati, f"src/generation.py non trovato. Radice: {[p.name for p in DEST.iterdir()][:10]}"
REPO = trovati[0].parent.parent
print("Radice del repo:", REPO)

# config.py solleva ValueError se ORACLE_API_KEY manca. Qui l'oracolo Groq non
# serve (genera solo le triplette di training), ma l'import va soddisfatto.
os.environ["ORACLE_API_KEY"] = "non-usata-in-valutazione"

# Verifica che ci sia tutto il necessario
print("  src/                 ", len(list((REPO / "src").glob("*.py"))), "file .py")
print("  data/processed/chunks ", len(list((REPO / "data/processed/chunks").rglob("*.jsonl"))), "file .jsonl")
print("  data/eval_questions   ", (REPO / "data/eval_questions.json").exists())
print("  config.py             ", (REPO / "config.py").exists())

# Stato del lavoro gia' fatto: se entrambi sono True puoi saltare le celle 5 e 7
print()
print("  patch gia' applicate  ", 'device="cuda"' in (REPO / "src/retrieval.py").read_text(encoding="utf-8"))
print("  indice gia' costruito ", (REPO / "data/vector_db").exists())

## 5 — Patch dei sorgenti

Modifica la copia in `/content/repo`, non il repository originale. Tre interventi:
encoder su GPU, generazione deterministica, giudice e bracci da valutare.

In [ ]:
import re

def patch(path, subs):
    p = REPO / path
    s = p.read_text(encoding="utf-8")
    for pattern, repl, attesi in subs:
        s, n = re.subn(pattern, repl, s)
        stato = "OK " if n == attesi else "!! "
        print(f"{stato}{path:22s} {n}/{attesi}  {pattern[:45]}")
    p.write_text(s, encoding="utf-8")

# 1) encoder su GPU
patch("src/retrieval.py", [
    (r'model_name="BAAI/bge-m3",\s*\n\s*device="cpu"[^\n]*',
     f'model_name="BAAI/bge-m3",\n            device="{EMBED_DEVICE}"', 1),
    (r'"BAAI/bge-reranker-v2-m3",\s*\n\s*device="cpu"[^\n]*',
     f'"BAAI/bge-reranker-v2-m3",\n            device="{RERANK_DEVICE}"', 1),
])

# 2) generazione deterministica (4 punti di chiamata: traduzione, generazione,
#    rigenerazione dei guardrail, baseline senza RAG)
patch("src/generation.py", [
    (r'temperature=0\.1,\s*',
     f'temperature={GEN_TEMPERATURE}, seed={GEN_SEED},\n                ', 4),
])

# 3) giudice e bracci
patch("src/evaluation.py", [
    (r'JUDGE_MODEL = "[^"]+"', f'JUDGE_MODEL = "{JUDGE_MODEL}"', 1),
    (r'MODELS = \[[^\]]+\]', "MODELS = " + repr(MODELS), 1),
])

In [ ]:
# Controllo visivo delle patch
!grep -n 'device=' {REPO}/src/retrieval.py | head -4
!grep -n 'temperature=\|seed=' {REPO}/src/generation.py | head -8
!grep -n 'JUDGE_MODEL =\|^MODELS' {REPO}/src/evaluation.py

## 6 — Modelli su Ollama

Scarica la baseline e il giudice dal registry, e ricrea i due modelli DPO dai GGUF su Drive.

In [ ]:
!ollama pull qwen2.5:3b
!ollama pull {JUDGE_MODEL}

In [ ]:
# Stesso template del Modelfile prodotto dal training (chat template di Qwen2.5).
# Costruito riga per riga per non annidare triple virgolette dentro la cella.
Q3 = '"' * 3
TEMPLATE = "\n".join([
    "TEMPLATE " + Q3 + "{{ if .System }}<|im_start|>system",
    "{{ .System }}<|im_end|>",
    "{{ end }}{{ if .Prompt }}<|im_start|>user",
    "{{ .Prompt }}<|im_end|>",
    "{{ end }}<|im_start|>assistant",
    "{{ .Response }}<|im_end|>",
    Q3,
    'PARAMETER stop "<|im_start|>"',
    'PARAMETER stop "<|im_end|>"',
    "",
])

for nome, gguf in [("astrotutor-dpo", GGUF_OLD), ("astrotutor-dpo-v2", GGUF_V2)]:
    mf = Path(f"/content/Modelfile.{nome}")
    mf.write_text(f"FROM {gguf}\n\n{TEMPLATE}", encoding="utf-8")
    print(f"\n--- {nome} ---")
    !ollama create {nome} -f {mf}

!ollama list

## 7 — Ricostruzione dell'indice ChromaDB

Rigenera il database vettoriale dai chunk presenti nel repo (`data/processed/chunks/`,
82 file JSONL). Non serve caricare i 263 MB dell'indice: si ricalcola qui, su GPU.

In [ ]:
%cd {REPO}
!python src/indexing.py

## 8 — Smoke test

Due domande e un giudizio, per verificare che la catena funzioni **prima** di lanciare
l'ora intera. Controlla che: il retrieval trovi documenti, la risposta sia sensata, e il
giudice restituisca un punteggio.

In [ ]:
import sys
from unittest.mock import MagicMock

# Fix per ModuleNotFoundError: langchain_community.chat_models.vertexai
# Ragas 0.4.3 ha un import non protetto che fallisce con le nuove versioni di langchain-community.
try:
    import langchain_community.chat_models.vertexai
except ImportError:
    sys.modules["langchain_community.chat_models.vertexai"] = MagicMock()
    sys.modules["langchain_community.llms.vertexai"] = MagicMock()

%cd {REPO}
import sys, asyncio
sys.path.insert(0, str(REPO))
import importlib, src.generation, src.evaluation
importlib.reload(src.generation)
importlib.reload(src.evaluation)

from src.generation import RAGGenerator
from src.evaluation import generate_with_docs
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness

rag = RAGGenerator(model_name="astrotutor-dpo-v2")
answer, docs = generate_with_docs(rag, "Cos'e' l'orizzonte degli eventi?", "A")
print(f"\nDocumenti recuperati: {len(docs)}")
print(f"Risposta ({len(answer)} char):\n{answer[:400]}\n")

judge = llm_factory(JUDGE_MODEL, provider="openai",
                    client=AsyncOpenAI(base_url="http://localhost:11434/v1",
                                       api_key="ollama", timeout=900.0),
                    max_tokens=8192)
score = await Faithfulness(llm=judge).ascore(
    user_input="Cos'e' l'orizzonte degli eventi?", response=answer,
    retrieved_contexts=[d["text"] for d in docs])
print(f"Faithfulness: {score.value:.3f}")

In [ ]:
# Determinismo: la stessa domanda deve dare la stessa identica risposta
a1, _ = generate_with_docs(rag, "Cosa succede alla materia in un buco nero?", "B")
a2, _ = generate_with_docs(rag, "Cosa succede alla materia in un buco nero?", "B")
print("Risposte identiche:", a1 == a2)
if a1 != a2:
    print("!! temperature/seed non stanno avendo effetto: ricontrolla la cella 5")

## 9 — Run completa

120 domande con contesto (40 × 3 modelli) + 60 domande OOD. ~1 ora.

Salva i risultati parziali su disco a ogni domanda: se il runtime si disconnette, rilancia
questa cella e riprende da dove si era fermata.

In [ ]:
%cd {REPO}
!python src/evaluation.py

## 10 — Salvataggio dei risultati su Drive

I file di progresso servono per rigiudicare in futuro senza rigenerare nulla: contengono
domanda, risposta, contesti e punteggi.

In [ ]:
import shutil
from datetime import datetime

dest = DRIVE / "results" / f"eval_{JUDGE_MODEL.replace(':','_')}_{datetime.now():%Y%m%d_%H%M}"
dest.mkdir(parents=True, exist_ok=True)

for f in (REPO / "data").glob("eval_*"):
    shutil.copy(f, dest / f.name)
    print(f"{f.name:50s} {f.stat().st_size/1e3:8.0f} KB")

print(f"\nSalvato in: {dest}")